# W5D5 — Fine-Tuning BERT — Guided

**Week 5 · Day 5 · NLP Foundations** · Lab

The payoff lab. Monday's TF-IDF baseline is on disk with its folds pinned. Today a model that was
pretrained on several billion words gets the same data, the same split, and a column for what it
costs.

Two eras of NLP, one table. Accuracy, F1, training time, inference time per thousand reviews, and
size on disk — and then the row that makes the whole week concrete: the negation pair from Monday,
classified by both.

The last task is not a coding task. Given a stated deployment constraint — 50 ms per request, CPU
only, no GPU budget — you write the paragraph saying which model you would ship. There is a
defensible answer either way, and the reasoning is the graded part. That paragraph is also the
second half of A5.

<div dir="rtl" align="right">

# الأسبوع ٥ · اليوم ٥ — الضبط الدقيق لـBERT

**الأسبوع الخامس · اليوم الخامس · أساسيات معالجة اللغة** · معمل

معمل المكسب. فأساس TF-IDF يوم الاثنين على القرص وأثلامه مثبّتة. واليوم يأخذ نموذجٌ دُرِّب مسبقًا على
عدّة مليارات كلمة البياناتِ نفسها والتقسيمَ نفسه وعمودًا لما يُكلّف.

عصران من معالجة اللغة في جدولٍ واحد. الدقّة ومقياس F1 وزمن التدريب وزمن الاستدلال لكل ألف مراجعة
والحجم على القرص — ثم الصف الذي يجعل الأسبوع كله محسوسًا: زوج النفي من الاثنين، مُصنَّفًا بالنموذجين.

والمهمة الأخيرة ليست مهمة برمجة. فبقيدٍ نشرٍ مُعلَن — خمسون ميلي ثانية للطلب، معالجٌ فقط، بلا ميزانية
معالج رسومات — تكتب الفقرة التي تقول أيّ نموذجٍ تنشر. والجواب قابل للدفاع في الحالتين، والتعليل هو
الجزء المُقيَّم. وتلك الفقرة هي أيضًا النصف الثاني من التكليف الخامس.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Read a subword tokenisation and say why the token count is not the word count.
- Explain what `pipeline("fill-mask")` is showing you, and connect it to pretraining.
- Assert that a tokeniser and a model come from the same checkpoint, and say what goes wrong if not.
- Fine-tune a pretrained encoder for one epoch on a CPU inside a classroom slot.
- Evaluate a new model on a split defined by an earlier lab, and verify it is the same split.
- Produce a model comparison with a cost column, and defend a deployment choice against a
  stated constraint.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تقرأ تقسيمًا جزئيًا وأن تقول لماذا لا يكون عدد الرموز عدد الكلمات.
- أن تشرح ما يُظهره لك `pipeline("fill-mask")`، وأن تصله بالتدريب المسبق.
- أن تفحص أن المُقسِّم والنموذج من نقطة التحقّق نفسها، وأن تقول ما يفسد إن لم يكونا كذلك.
- أن تضبط مُرمِّزًا مُدرَّبًا مسبقًا حقبةً واحدة على معالجٍ داخل حصّة القاعة.
- أن تُقيّم نموذجًا جديدًا على تقسيمٍ حدّده معمل سابق، وأن تتحقّق أنه التقسيم نفسه.
- أن تُخرج مقارنة نماذج فيها عمود كلفة، وأن تدافع عن اختيار نشرٍ مقابل قيدٍ مُعلَن.

</div>


## About the data

**Dataset:** `reviews_sentiment` — the same 12,000 reviews as Monday. That is what makes today a
comparison rather than an anecdote.

**Read this paragraph before you run anything.** Fine-tuning is subsampled to **2,000 reviews**,
truncated at `max_length=128`, for **one epoch**. That takes **one to two minutes on Apple
silicon and up to about six on an older classroom CPU**, during which the cell prints a loss every
25 steps and appears to do nothing else. It has not hung. Do not interrupt it. If you do, the model is left half-trained and the comparison at the
end of the lab is meaningless.

**First-run download:** `distilbert-base-uncased`, about 260 MB, cached afterwards. The stretch
section loads four more pipelines and a multilingual checkpoint, which is another 1–2 GB — skip the
stretch if the room's connection is struggling.

**The evaluation split is not ours to choose.** It is Monday's fold 0, reconstructed from
`tfidf_baseline.json`, and task 2.4 asserts that the reconstruction hashes to the same value the
artefact recorded. A comparison on two different splits is the most common way this exact table gets
published wrong.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `reviews_sentiment` — المراجعات الاثنتا عشرة ألفًا نفسها يوم الاثنين. وهذا ما
يجعل اليوم مقارنةً لا حكاية.

**اقرأ هذه الفقرة قبل أن تُشغّل شيئًا.** الضبط الدقيق على عيّنةٍ من **ألفَي مراجعة**، مقطوعةً عند
`max_length=128`، لـ**حقبةٍ واحدة**. ويأخذ ذلك **دقيقةً أو دقيقتين على معالجات آبل وإلى نحو ست دقائق
على معالج قاعةٍ أقدم**، تطبع الخليّة خلالها خسارةً كل خمسٍ وعشرين خطوة ويبدو أنها لا تفعل غير ذلك. وهي لم تتعلّق. فلا تقطعها. وإن قطعتها
بقي النموذج نصف مُدرَّب وصارت المقارنة في نهاية المعمل بلا معنى.

**تنزيل أول مرّة:** `distilbert-base-uncased`، نحو ٢٦٠ ميغابايت، ثم يُخزَّن. ويُحمّل قسم التمديد
أربعة خطوط معالجة أخرى ونقطة تحقّقٍ متعدّدة اللغات، وهي جيغابايت أو اثنان أخرى — فتجاوز التمديد إن
كان اتّصال القاعة يتعثّر.

**وتقسيم التقييم ليس لنا أن نختاره.** بل هو الثلم صفر يوم الاثنين، مُعاد بناؤه من
`tfidf_baseline.json`، وتفحص المهمة ٢٫٤ أن إعادة البناء تُجزَّأ إلى القيمة نفسها التي سجّلها الأثر.
والمقارنة على تقسيمين مختلفين أشيع طريقةٍ يُنشر بها هذا الجدول بعينه خاطئًا.

</div>


## Setup

The training loop here is plain PyTorch — a `DataLoader`, `AdamW`, and a loop you can read — rather
than `transformers.Trainer`. Two reasons: the loop is eleven lines and you have written it twice
already this course, and it does not change between library versions, which matters for a notebook
that has to run in a year's time.

<div dir="rtl" align="right">

## الإعداد

حلقة التدريب هنا PyTorch صريح — `DataLoader` و`AdamW` وحلقةٌ تستطيع قراءتها — لا
`transformers.Trainer`. ولسببين: الحلقة أحد عشر سطرًا وقد كتبتها مرّتين في هذه الدورة أصلًا، وهي لا
تتغيّر بين إصدارات المكتبة، وهذا مهمّ لدفترٍ عليه أن يعمل بعد سنة.

</div>


In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, describe_dataset, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report
from aiep.viz import use_course_style, savefig

ensure("scikit-learn", "matplotlib", "transformers", "torch")
seed_everything(42)

import hashlib
import json
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

use_course_style()

SEED = 42
CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 1
LEARNING_RATE = 2e-5
TRAIN_SUBSET = 2_000
LOG_EVERY = 25

NEGATION_PAIR = ["the service was good", "the service was not good"]

reviews = pd.read_parquet(get_dataset("reviews_sentiment"))
baseline = json.loads(load_artefact("tfidf_baseline.json").read_text(encoding="utf-8"))

print(describe_dataset("reviews_sentiment"))
print(f"\nMonday's baseline: {baseline['accuracy_mean']:.4f} ± {baseline['accuracy_std']:.4f} "
      f"over {baseline['cv']['n_splits']} folds, fitted in {baseline['fit_seconds']:.1f}s")
print(f"fold 0 alone     : {baseline['accuracy_folds'][0]:.4f}  ← today's number to beat")
print(versions(), "| device:", device())

## Section 1 — Warm-up: pretraining, visible in five lines  (≈25 min)

Everything here works, and none of it trains anything.

**One.** Tokenise `"the service was not good"`. It comes back as **7** tokens, not 5:
`[CLS] the service was not good [SEP]`. The two extra are structural — `[CLS]` is the slot whose
final vector becomes the classification input, and `[SEP]` marks the end. Every length calculation
in this lab counts them.

**Two.** Tokenise `unhappiness`. One word, **three** subword pieces, the continuation ones marked
with `##`. That is why the model has no unknown words: anything it has never seen is spelled out of
pieces it has. It is also why the token count is never the word count.

**Three.** `pipeline("fill-mask")` on `"the service was [MASK]"`, printing the top five. Nothing was
trained here, by you or for you. That output *is* the pretraining — a model that read a large amount
of text and learned which word goes in a hole. Fine-tuning, for the next hour, is nothing but
attaching a small head to that and nudging the weights.

Change the masked sentence and see how much the top five moves.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: التدريب المسبق مرئيًا في خمسة أسطر (نحو ٢٥ دقيقة)

كل ما هنا يعمل، ولا شيء منه يُدرِّب شيئًا.

**أولًا.** قسّم `"the service was not good"`. فتعود **سبعة** رموز لا خمسة:
`[CLS] the service was not good [SEP]`. والزائدان بنيويّان — فـ`[CLS]` هو الخانة التي يصير متّجهها
النهائي دخلَ التصنيف، و`[SEP]` تُعلّم النهاية. وكل حساب طولٍ في هذا المعمل يعدّهما.

**ثانيًا.** قسّم `unhappiness`. كلمة واحدة و**ثلاث** قطع جزئية، والمُتابِعة منها مُعلَّمة بـ`##`.
ولهذا لا كلمات مجهولة عند النموذج: فكل ما لم يره يُهجّأ من قطعٍ عنده. ولهذا أيضًا لا يكون عدد الرموز
عدد الكلمات أبدًا.

**ثالثًا.** `pipeline("fill-mask")` على `"the service was [MASK]"`، طابعًا أفضل خمسة. ولم يُدرَّب
هنا شيء، لا بك ولا لك. فذلك الخرج **هو** التدريب المسبق — نموذجٌ قرأ قدرًا كبيرًا من النص وتعلّم أي
كلمةٍ تصلح في ثقب. والضبط الدقيق في الساعة القادمة ليس إلا إلحاق رأسٍ صغير بذلك وإزاحة الأوزان.

غيّر الجملة المُقنَّعة وانظر كم يتحرّك أفضل خمسة.

</div>


In [ ]:
from transformers import AutoTokenizer, pipeline

tokeniser = AutoTokenizer.from_pretrained(CHECKPOINT)

encoded = tokeniser(NEGATION_PAIR[1])
tokens = tokeniser.convert_ids_to_tokens(encoded["input_ids"])
NEGATION_TOKEN_COUNT = len(tokens)

print(f"{NEGATION_PAIR[1]!r}")
print(f"  words : {len(NEGATION_PAIR[1].split())}")
print(f"  tokens: {NEGATION_TOKEN_COUNT}  {tokens}")
print(f"  ids   : {encoded['input_ids']}")

subwords = tokeniser.tokenize("unhappiness")
print(f"\n'unhappiness' -> {subwords}  ({len(subwords)} pieces, '##' marks a continuation)")
print(f"'antidisestablishmentarianism' -> "
      f"{len(tokeniser.tokenize('antidisestablishmentarianism'))} pieces — nothing is unknown, "
      f"everything is spelled")

In [ ]:
filler = pipeline("fill-mask", model=CHECKPOINT)
predictions = filler(f"the service was {filler.tokenizer.mask_token}.", top_k=5)

print(f"the service was [MASK].\n")
for prediction in predictions:
    print(f"  {prediction['token_str']:>12}  {prediction['score']:.3f}")
print("\nNo training happened. This is what the checkpoint already knew, and it is the only "
      "reason the next hour works on 2,000 reviews instead of 2,000,000.")

## Section 2 — Core: six tasks  (≈60 min)

1. Load the tokeniser and the model, and assert they are the same checkpoint.
2. Tokenise the subsample at 128, and report what fraction got truncated.
3. Fine-tune for one epoch, logging the loss.
4. Evaluate on **Monday's fold 0**, verified by hash.
5. The comparison table, with both time columns, written to `comparison.md`.
6. The recommendation, against a stated constraint.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. حمّل المُقسِّم والنموذج، وافحص أنهما من نقطة التحقّق نفسها.
٢. قسّم العيّنة عند ١٢٨، واعرض ما نسبة ما قُطع.
٣. اضبط حقبةً واحدة مع تسجيل الخسارة.
٤. قيّم على **الثلم صفر يوم الاثنين**، مُتحقَّقًا منه بالتجزيء.
٥. جدول المقارنة، بعمودَي الزمن كليهما، مكتوبًا في `comparison.md`.
٦. التوصية، مقابل قيدٍ مُعلَن.

</div>


### Task 2.1 — same checkpoint, asserted

Load `AutoModelForSequenceClassification` with `num_labels=2`, and note the warning it prints: the
classification head is **newly initialised**, because the pretrained checkpoint never had one. That
warning is correct and expected. The one that is not expected is silence when you have mixed a
tokeniser from one checkpoint with a model from another.

Assert that they match, by comparing `tokeniser.name_or_path` against `model.config._name_or_path`.

This is the "silent garbage" failure from the lecture, and it deserves the sentence: a tokeniser maps
text to integers using **its own** vocabulary. Hand those integers to a model trained on a different
vocabulary and nothing raises — every id is in range, every shape is right, and the model reads
your sentence as a different sentence. The accuracy comes out somewhere above chance and below
useful, and there is nothing in the traceback to look at, because there is no traceback.

<div dir="rtl" align="right">

### المهمة ٢٫١ — نقطة التحقّق نفسها، مفحوصةً

حمّل `AutoModelForSequenceClassification` بـ`num_labels=2`، ولاحظ التحذير الذي يطبعه: فرأس التصنيف
**مُهيّأ حديثًا**، لأن نقطة التحقّق المُدرَّبة مسبقًا لم يكن لها رأس. وذلك التحذير صحيح ومتوقّع. أما
غير المتوقّع فالصمت حين تخلط مُقسِّمًا من نقطة تحقّقٍ بنموذجٍ من أخرى.

افحص تطابقهما بمقارنة `tokeniser.name_or_path` بـ`model.config._name_or_path`.

وهذا هو إخفاق «القمامة الصامتة» من المحاضرة، ويستحقّ الجملة: فالمُقسِّم يُحوّل النص أعدادًا صحيحة
بمعجمه **هو**. وسلّم تلك الأعداد نموذجًا دُرِّب على معجمٍ آخر فلا يرتفع شيء — فكل معرّف في المدى، وكل
شكلٍ صحيح، ويقرأ النموذج جملتك جملةً أخرى. فتخرج الدقّة فوق الحظّ ودون النافع، ولا شيء في تتبّع
الاستثناء لتنظر فيه، لأن لا تتبّع استثناء.

</div>


In [ ]:
from transformers import AutoModelForSequenceClassification

# TODO: checkpoint, and print the total parameter count against the size of the new head.
# مهمة: واطبع عدد المعاملات الكلي مقابل حجم الرأس الجديد.

### Task 2.2 — tokenise at 128, and count what you threw away

Take the training subsample, tokenise it with `truncation=True, max_length=128, padding="max_length"`,
and then answer the question nobody asks: **what fraction of the reviews did not fit?**

Tokenise once without truncation to get the true lengths, and report the fraction above 128 and the
median length. On this corpus a meaningful slice gets cut, and every cut review loses its ending —
which in a review is often exactly where the verdict is ("...but overall I'd buy it again").

Write the sentence on what that costs. Then note what it buys: attention is quadratic in sequence
length, so 128 instead of 512 is roughly a sixteenth of the attention arithmetic, which is the
difference between six minutes and an afternoon.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — قسّم عند ١٢٨ وعُدّ ما رميته

خُذ عيّنة التدريب، وقسّمها بـ`truncation=True, max_length=128, padding="max_length"`، ثم أجب السؤال
الذي لا يسأله أحد: **ما نسبة المراجعات التي لم تتّسع؟**

قسّم مرّةً بلا قطعٍ لتحصل على الأطوال الحقيقية، واعرض نسبة ما فوق ١٢٨ والطول الوسيط. وعلى هذه
المُدوّنة تُقطع شريحةٌ معتبرة، وكل مراجعة مقطوعة تفقد نهايتها — وهي في المراجعة موضع الحكم غالبًا
(«...لكنني على العموم سأشتريه مرّةً أخرى»).

اكتب الجملة عمّا يُكلّفه ذلك. ثم لاحظ ما يكسبه: فالانتباه تربيعيّ في طول المتتالية، فتكون ١٢٨ بدل
٥١٢ نحو سُدس عشر حساب الانتباه، وهذا هو الفرق بين ست دقائق وظهيرة.

</div>


In [ ]:
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, TensorDataset

# Monday's splitter, rebuilt from the artefact rather than retyped.
cv = StratifiedKFold(n_splits=baseline["cv"]["n_splits"],
                     shuffle=baseline["cv"]["shuffle"],
                     random_state=baseline["cv"]["random_state"])
FOLDS = [test.tolist() for _, test in cv.split(reviews.review_text, reviews.sentiment)]
train_pool, EVAL_INDEX = np.array(sorted(set(range(len(reviews))) - set(FOLDS[0]))), \
    np.array(FOLDS[0])

rng = np.random.default_rng(SEED)
TRAIN_INDEX = np.sort(rng.choice(train_pool, size=TRAIN_SUBSET, replace=False))
print(f"train on {len(TRAIN_INDEX):,} sampled from fold 0's training rows, "
      f"evaluate on all {len(EVAL_INDEX):,} of fold 0")

# TODO: fraction with the median and maximum, then tokenise for training and build the loader.
# مهمة: للتدريب وابنِ المُحمّل.

### Task 2.3 — one epoch, and the loss on the way down

The loop: for each batch, forward, take `outputs.loss`, backward, step, zero the gradients. Eleven
lines, and you have written it in week 3 and again in week 4. What is different is only that the
model already knows English.

Log the loss every 25 steps and plot it at the end. Two things to look for:

- The loss starts near `ln(2) ≈ 0.69`, which is what a two-class model that knows nothing outputs,
  and drops fast. It is not learning English in that first minute — it is learning what the
  freshly-initialised head is for.
- **A single batch's loss is useless as a trend.** Consecutive steps here swing by half a nat —
  0.42 then 0.71 then 0.16 — because sixteen reviews is a tiny sample of a noisy objective. So the
  cell logs the **mean over each window of 25 steps**, and the sanity check at the end compares the
  first window against the last. Reading a trend off two individual steps is how you convince
  yourself a working run is broken.

**This cell takes one to two minutes on Apple silicon and up to about six on an older classroom
CPU.** It is not stuck.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — حقبةٌ واحدة، والخسارة في هبوطها

الحلقة: لكل دفعة، تمريرٌ أمامي، ثم خُذ `outputs.loss`، ثم تمريرٌ خلفي، ثم خطوة، ثم صفّر التدرّجات.
أحد عشر سطرًا، وقد كتبتها في الأسبوع الثالث ثم الرابع. والفرق الوحيد أن النموذج يعرف الإنجليزية
أصلًا.

سجّل الخسارة كل خمسٍ وعشرين خطوة وارسمها في النهاية. وشيئان يجدر النظر إليهما:

- تبدأ الخسارة قرب `ln(2) ≈ 0.69`، وهو ما يُخرجه نموذج فئتين لا يعرف شيئًا، وتهبط سريعًا. وهو لا
  يتعلّم الإنجليزية في تلك الدقيقة الأولى — بل يتعلّم لماذا وُجد الرأس المُهيّأ حديثًا.
- **وخسارة دفعةٍ واحدة لا تصلح اتّجاهًا.** فالخطوات المتتالية هنا تتأرجح نصف نات — ٠٫٤٢ ثم ٠٫٧١
  ثم ٠٫١٦ — لأن ست عشرة مراجعة عيّنةٌ ضئيلة من دالّة هدفٍ مُشوَّشة. فتسجّل الخليّة **المتوسط على كل
  نافذةٍ من خمسٍ وعشرين خطوة**، ويقارن فحص السلامة في النهاية النافذة الأولى بالأخيرة. وقراءة
  الاتّجاه من خطوتين منفردتين هي طريق إقناع نفسك أن تشغيلةً سليمة مكسورة.

**وتأخذ هذه الخليّة دقيقةً أو دقيقتين على معالجات آبل وإلى نحو ست دقائق على معالج قاعةٍ أقدم.** وهي
ليست متعلّقة.

</div>


In [ ]:
# TODO: record the total wall-clock in FIT_SECONDS.
# مهمة: الكلي في `FIT_SECONDS`.

### Task 2.4 — evaluate on Monday's fold, and prove it is Monday's fold

Score the fine-tuned model on `EVAL_INDEX` — fold 0 of Monday's cross-validation, 2,400 reviews the
model has never seen.

Before you report anything, **verify the split**. Hash the reconstructed fold indices exactly as
Monday hashed them and compare against `baseline["cv"]["folds_sha256"]`. If the hashes differ, stop:
the two models are being measured on different data and every number below is noise dressed as a
result.

Then fit Monday's TF-IDF pipeline on the *same* 2,000 training rows and score it on the *same* 2,400
evaluation rows, so the table compares representations rather than dataset sizes. Record accuracy,
F1, and — separately — the time to score 1,000 reviews with each. Also record the negation pair for
both models.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — قيّم على ثلم الاثنين، وبرهِن أنه ثلم الاثنين

قيّم النموذج المضبوط على `EVAL_INDEX` — الثلم صفر من تحقّق الاثنين التقاطعي، ألفان وأربعمئة مراجعة
لم يرها النموذج قطّ.

وقبل أن تعرض شيئًا، **تحقّق من التقسيم**. جزّئ فهارس الأثلام المُعاد بناؤها كما جزّأها الاثنين
بالضبط وقارن بـ`baseline["cv"]["folds_sha256"]`. فإن اختلف التجزيئان فتوقّف: فالنموذجان يُقاسان على
بيانات مختلفة وكل رقمٍ أدناه ضجيجٌ في ثوب نتيجة.

ثم درّب خطّ TF-IDF يوم الاثنين على صفوف التدريب الألفين **نفسها** وقيّمه على صفوف التقييم الألفين
وأربعمئة **نفسها**، ليقارن الجدول التمثيلات لا أحجام البيانات. وسجّل الدقّة ومقياس F1 و — على حدة —
زمن تصنيف ألف مراجعة بكلٍّ منهما. وسجّل زوج النفي للنموذجين أيضًا.

</div>


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline

eval_texts = reviews.review_text.iloc[EVAL_INDEX].tolist()
eval_labels = reviews.sentiment.iloc[EVAL_INDEX].to_numpy()

# TODO: Verify the fold hash against Monday's artefact before reporting anything.
# مهمة: تحقّق من تجزيء الأثلام مقابل أثر الاثنين قبل عرض أي شيء.

# TODO: pipeline on the same rows, and time 1,000 predictions with each.
# مهمة: نفسها، ووقّت ألف تنبّؤ بكلٍّ منهما.

### Task 2.5 — the table, with the cost in it

Save the model to `bert_finetuned/`, measure what it takes up on disk, do the same for the TF-IDF
pipeline with `joblib`, and write `comparison.md` with five columns for both models: **accuracy,
F1, fit time, inference time per 1,000 reviews, size on disk.**

All five. A comparison with only the first two is the failure this lab exists to prevent — it is the
table in every blog post that concludes "BERT wins", omitting that it wins by a couple of points
while costing two orders of magnitude more per request and a hundred times the disk.

Then add the negation row. `"the service was not good"`, classified by both. One row, no metrics,
and it is the clearest thing in the file.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — الجدول والكلفة فيه

احفظ النموذج في `bert_finetuned/`، وقِس ما يشغله على القرص، وافعل الشيء نفسه لخطّ TF-IDF بـ`joblib`،
واكتب `comparison.md` بخمسة أعمدة للنموذجين: **الدقّة ومقياس F1 وزمن التدريب وزمن الاستدلال لكل ألف
مراجعة والحجم على القرص.**

الخمسة كلها. فالمقارنة بالأولين وحدهما هي الفشل الذي وُجد هذا المعمل لمنعه — وهي الجدول في كل تدوينة
تستنتج أن «BERT يفوز»، مُهمِلةً أنه يفوز بنقطتين وهو يُكلّف مرتبتين أعلى للطلب ومئة ضعفٍ من القرص.

ثم أضِف صف النفي. `"the service was not good"` مُصنَّفةً بالنموذجين. صفٌّ واحد بلا مقاييس، وهو أوضح
ما في الملف.

</div>


In [ ]:
import joblib

BERT_DIR = ARTEFACT_DIR / "bert_finetuned"
TFIDF_PATH = ARTEFACT_DIR / "tfidf_pipeline.joblib"

# TODO: write comparison.md with all five columns for both models plus the negation row.
# مهمة: `comparison.md` بالأعمدة الخمسة للنموذجين ومعها صف النفي.

### Task 2.6 — the recommendation

Not a coding task. Here is the constraint, and it is the kind you will actually be handed:

> The service must answer in **under 50 ms per request** on a **CPU**. There is no GPU budget. It
> classifies about 200,000 reviews a day, and the accuracy target is "as good as we can get inside
> the latency budget".

Use the **single-request** timing, not the batched one. Task 2.4 measured both, and they differ by
an order of magnitude: batching 32 reviews spreads the per-call overhead across all 32, and a
service answering one request at a time never gets that discount. Reporting throughput where the
requirement says latency is the most common way this table gets misread.

Put that number next to the 50 ms, and write the paragraph: **which model do you ship, and why?**

Both answers are defensible, and which one your numbers point at depends on your machine — on a
recent laptop BERT comes in around 10 ms and comfortably meets the budget, on an older CPU it does
not. Shipping TF-IDF is defensible if the latency figure says BERT cannot meet the budget.
Shipping BERT is defensible if you name what makes it safe: a load test at the real concurrency,
the daily compute bill, the 270 MB to deploy, and a fallback for when the model server is down.

What is **not** defensible is picking the higher accuracy without mentioning the constraint. That is
the entire skill this task is testing, and it is graded on the reasoning, not the choice.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — التوصية

ليست مهمة برمجة. وهذا هو القيد، وهو من النوع الذي ستُسلَّمه فعلًا:

> يجب أن يُجيب الخدمة في **أقلّ من خمسين ميلي ثانية للطلب** على **معالج**. ولا ميزانية لمعالج
> رسومات. وهي تُصنّف نحو مئتَي ألف مراجعة يوميًا، وهدف الدقّة «أفضل ما نبلغه داخل ميزانية الكمون».

استخدم توقيت الطلب **الواحد** لا المُدفَّع. فقد قاست المهمة ٢٫٤ الاثنين، وهما يختلفان بمرتبة: إذ
يُوزّع تدفيع اثنتين وثلاثين مراجعة عبء النداء على الاثنتين والثلاثين كلها، والخدمة التي تُجيب طلبًا
واحدًا في كل مرّة لا تنال ذلك الخصم قطّ. وعرض السَّعة حيث يقول الشرط «كمون» أشيع طريقةٍ يُقرأ بها
هذا الجدول قراءةً خاطئة.

ضع ذلك الرقم بجانب الخمسين ميلي ثانية، واكتب الفقرة: **أيّ نموذج تنشر، ولماذا؟**

والجوابان قابلان للدفاع، وأيّهما تشير إليه أرقامك يتوقّف على جهازك — فعلى حاسبٍ محمول حديث يأتي
BERT نحو عشر ميلي ثوانٍ ويفي بالميزانية مرتاحًا، وعلى معالجٍ أقدم لا يفي. فنشر TF-IDF قابل للدفاع
إن قال رقم الكمون إن BERT لا يستطيع الوفاء بالميزانية. ونشر BERT قابل للدفاع إن سمّيت ما يجعله
آمنًا: اختبار حملٍ على التزامن الحقيقي، وفاتورة الحساب اليومية، والمئتان وسبعون ميغابايت التي
تُنشَر، وطريقٌ بديل حين يسقط خادم النموذج.

وغير القابل للدفاع اختيار الدقّة الأعلى بلا ذكر القيد. وهذه هي المهارة التي تفحصها هذه المهمة كلها،
وهي مُقيَّمة على التعليل لا على الاختيار.

</div>


In [ ]:
LATENCY_BUDGET_MS = 50
DAILY_VOLUME = 200_000

# TODO: recommendation paragraph into RECOMMENDATION.
# مهمة: `RECOMMENDATION`.

## Section 3 — Stretch: four pipelines, and the 4% again  (≈30 min)

**(a) The applied survey.** One paragraph, four `pipeline` calls. One sentence each on where you
would use it — and one on **how it failed**, because each of them will, visibly, on a paragraph this
short. That second sentence is the useful half of the exercise.

**A correction to the lab spec, worth knowing before you search for it.** The spec asked for `ner`,
`question-answering`, `summarization` and `translation`. `transformers` 5 removed the last three as
pipeline aliases — `pipeline("summarization")` now raises a `KeyError`, and those tasks are reached
either through an instruction-tuned model under `text-generation` or through the task-specific
`AutoModelFor…` classes directly. `ner` survives as an alias for `token-classification`. So the four
here are `token-classification`, `zero-shot-classification`, `text-generation` and
`text-classification`, which is a better survey anyway: two of them are things you could not do at
all before pretraining.

**Each pipeline downloads its own model** — roughly 1 GB together. Skip this section without guilt
if the room's connection is struggling; nothing later depends on it.

**(b) The 4% you found on Monday.** Score the fine-tuned model on the Arabic subset separately, the
way D1's stretch did, and compare the two gaps. Then load a multilingual checkpoint and score the
same subset with it.

**The finding is not the one the registry's `gotcha` leads you to expect, and that is the task.**
The `gotcha` says "TF-IDF ignores them; BERT partly doesn't". On the reference run the fine-tuned
English BERT scores about **0.52** on the Arabic subset — chance — while D1's TF-IDF managed
**0.76**. BERT is worse, not better.

Before concluding that subword tokenisation does not help, count the Arabic reviews in each model's
*training* data. D1 cross-validated on all 12,000 rows, so each fold trained on roughly 380 Arabic
reviews. Today's 2,000-row subsample contains around 80. The comparison is not English-BERT against
English-TF-IDF; it is 80 examples against 380. Report the count alongside the accuracy, and note
that 93 Arabic rows in the evaluation fold puts the standard error around ±0.05 — wide enough that
the multilingual checkpoint's result needs the same caveat.

Then say what you would actually do about it. That answer is not "a bigger model".

<div dir="rtl" align="right">

## القسم الثالث — التمديد: أربعة خطوط، والأربعة بالمئة مرّةً أخرى (نحو ٣٠ دقيقة)

**(أ) المسح التطبيقي.** فقرةٌ واحدة وأربعة نداءات `pipeline`. وجملةٌ لكلٍّ منها عن موضع استخدامك له
— وجملةٌ عن **كيف فشل**، لأن كلًّا منها سيفشل ظاهرًا على فقرةٍ بهذا القِصَر. وتلك الجملة الثانية هي
النصف المُفيد من التمرين.

**وتصحيحٌ لمواصفة المعمل يجدر معرفته قبل أن تبحث عنه.** طلبت المواصفة `ner`
و`question-answering` و`summarization` و`translation`. وقد حذف `transformers` 5 الثلاثة الأخيرة
كأسماء خطوط — فـ`pipeline("summarization")` ترفع الآن `KeyError`، وتلك المهام تُنال إمّا بنموذجٍ
مُوجَّه بالتعليمات تحت `text-generation` وإمّا بأصناف `AutoModelFor…` المخصّصة مباشرةً. وبقي `ner`
اسمًا بديلًا لـ`token-classification`. فالأربعة هنا `token-classification`
و`zero-shot-classification` و`text-generation` و`text-classification`، وهي مسحٌ أفضل على أي حال:
فاثنان منها شيئان لم تكن تستطيعهما أصلًا قبل التدريب المسبق.

**وكل خطّ يُنزّل نموذجه** — نحو جيغابايت معًا. فتجاوز هذا القسم بلا حرج إن كان اتّصال القاعة
يتعثّر؛ ولا شيء بعده يتوقّف عليه.

**(ب) الأربعة بالمئة التي وجدتها الاثنين.** قيّم النموذج المضبوط على المجموعة العربية وحدها كما فعل
تمديد اليوم الأول، وقارن الفجوتين. ثم حمّل نقطة تحقّقٍ متعدّدة اللغات وقيّم المجموعة نفسها بها.

**والنتيجة ليست التي يقودك إليها تحذير السجلّ، وهذه هي المهمة.** يقول التحذير «يتجاهلها TF-IDF،
بينما لا يتجاهلها BERT كليًا». وفي التشغيلة المرجعية يسجّل BERT الإنجليزي المضبوط نحو **٠٫٥٢** على
المجموعة العربية — أي الحظّ — بينما بلغ TF-IDF في اليوم الأول **٠٫٧٦**. فـBERT أسوأ لا أفضل.

وقبل أن تستنتج أن التقسيم الجزئي لا يُفيد، عُدّ المراجعات العربية في بيانات **تدريب** كل نموذج. فقد
تحقّق اليوم الأول تقاطعيًا من الاثنتي عشرة ألف صف كلها، فتدرّب كل ثلمٍ على نحو ٣٨٠ مراجعة عربية.
وعيّنة اليوم من ألفَي صف فيها نحو ثمانين. فليست المقارنة بين BERT إنجليزي وTF-IDF إنجليزي، بل بين
ثمانين مثالًا وثلاثمئة وثمانين. اعرض العدد بجانب الدقّة، ولاحظ أن ثلاثًا وتسعين مراجعة عربية في ثلم
التقييم تجعل الخطأ المعياري نحو ±٠٫٠٥ — وهي سعةٌ تكفي لأن تحتاج نتيجة النموذج متعدّد اللغات التحفّظ
نفسه.

ثم قل ما كنت لتفعله فعلًا بشأنه. وذلك الجواب ليس «نموذجًا أكبر».

</div>


In [ ]:
PARAGRAPH = (
    "The charger arrived from Anker on Tuesday, two days later than the delivery estimate. "
    "It works with my laptop and both phones, and the two USB-C ports charge at full speed "
    "at the same time, which the older model could not do. The plastic housing gets warm "
    "under load but never hot. Support in Riyadh answered a question about the warranty "
    "within a day. I would buy it again, though I would order it a week earlier."
)
LABELS = ["delivery", "build quality", "customer support", "price"]
MULTILINGUAL = "nlptown/bert-base-multilingual-uncased-sentiment"

# TODO: it failed, then score the Arabic subset of the evaluation fold with both checkpoints.
# مهمة: قيّم المجموعة العربية من ثلم التقييم بنقطتَي التحقّق كلتيهما.

In [ ]:
# TODO: against D1's number.
# مهمة: النصف العربي بنقطة تحقّقٍ متعدّدة اللغات، واطبع ذلك كله مقابل رقم اليوم الأول.

## Save your artefact

Two things go to disk: `bert_finetuned/` — the model and tokeniser, gitignored, a few hundred
megabytes — and `comparison.md`, which is the file that matters.

`comparison.md` is A5's first half. The assignment is this comparison written up properly: the five
columns, the negation row, and the recommendation paragraph with its numbers in it. Do not rewrite
the numbers by hand into the assignment — read them out of the file, which is why the file exists.

<div dir="rtl" align="right">

## احفظ أثرك

شيئان يذهبان إلى القرص: `bert_finetuned/` — النموذج والمُقسِّم، مُستثنيان من git، بمئات
الميغابايتات — و`comparison.md`، وهو الملف المهمّ.

و`comparison.md` هو النصف الأول من التكليف الخامس. فالتكليف هذه المقارنة مكتوبةً كتابةً صحيحة:
الأعمدة الخمسة، وصف النفي، وفقرة التوصية والأرقام فيها. ولا تُعِد كتابة الأرقام يدويًا في التكليف —
بل اقرأها من الملف، ولهذا يوجد الملف.

</div>


In [ ]:
summary = {
    "dataset": "reviews_sentiment",
    "checkpoint": CHECKPOINT,
    "train_rows": int(len(TRAIN_INDEX)),
    "eval_rows": int(len(EVAL_INDEX)),
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "truncated_fraction": round(TRUNCATED_FRACTION, 4),
    "split": {"source": "D1 fold 0", "folds_sha256": FOLDS_SHA256, "matches_d1": SPLIT_MATCHES},
    "bert": {"accuracy": round(BERT_ACCURACY, 4), "f1": round(BERT_F1, 4),
             "fit_seconds": round(FIT_SECONDS, 2),
             "inference_seconds_per_1000": round(BERT_INFERENCE, 4),
             "single_request_ms": round(BERT_SINGLE_MS, 2),
             "disk_mb": round(BERT_DISK_MB, 1)},
    "tfidf": {"accuracy": round(TFIDF_ACCURACY, 4), "f1": round(TFIDF_F1, 4),
              "fit_seconds": round(TFIDF_FIT_SECONDS, 2),
              "inference_seconds_per_1000": round(TFIDF_INFERENCE, 4),
              "single_request_ms": round(TFIDF_SINGLE_MS, 3),
              "disk_mb": round(TFIDF_DISK_MB, 3)},
    "d1_full_corpus_accuracy": baseline["accuracy_mean"],
    "negation": NEGATION_RESULTS,
    "loss_log": LOSS_LOG,
    "recommendation": RECOMMENDATION,
}

SUMMARY_PATH = ARTEFACT_DIR / "bert_comparison.json"
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps({"bert": summary["bert"], "tfidf": summary["tfidf"],
                  "split": {"matches_d1": SPLIT_MATCHES}}, indent=2))
print(f"\nwrote {SUMMARY_PATH.name}, {COMPARISON_PATH.name} and {BERT_DIR.name}/")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>


In [ ]:
check(CHECKPOINTS_MATCH,
      f"the tokeniser and the model must come from the same checkpoint — got "
      f"{tokeniser.name_or_path!r} and {model.config._name_or_path!r}. Mixing them raises nothing "
      f"and produces a model that reads a different sentence than the one you gave it",
      f"يجب أن يكون المُقسِّم والنموذج من نقطة التحقّق نفسها — والناتج {tokeniser.name_or_path!r} و"
      f"{model.config._name_or_path!r}. وخلطهما لا يرفع شيئًا ويُخرج نموذجًا يقرأ جملةً غير التي "
      f"أعطيته")

check(NEGATION_TOKEN_COUNT == 7,
      f"'{NEGATION_PAIR[1]}' must tokenise to 7 tokens including [CLS] and [SEP], not 5 — got "
      f"{NEGATION_TOKEN_COUNT}. If it is 5 you asked for the tokens without the special ones, and "
      f"every length calculation below will be off by two",
      f"يجب أن تُقسَّم «{NEGATION_PAIR[1]}» إلى سبعة رموز مع `[CLS]` و`[SEP]` لا خمسة — والناتج "
      f"{NEGATION_TOKEN_COUNT}. فإن كان خمسة فقد طلبت الرموز بلا الخاصّة، وسيزيغ كل حساب طولٍ "
      f"أدناه باثنين")

check(SPLIT_MATCHES,
      f"the evaluation split must be byte-identical to D1's — rebuilt {FOLDS_SHA256[:16]}… "
      f"against the artefact's {baseline['cv']['folds_sha256'][:16]}…. If these differ the two "
      f"models were measured on different data and the whole comparison is void",
      f"يجب أن يكون تقسيم التقييم مطابقًا بايتًا ببايت لتقسيم اليوم الأول — المُعاد بناؤه "
      f"{FOLDS_SHA256[:16]}… مقابل {baseline['cv']['folds_sha256'][:16]}… في الأثر. فإن اختلفا "
      f"فقد قِيس النموذجان على بيانات مختلفة والمقارنة كلها باطلة")

check(0.0 <= TRUNCATED_FRACTION <= 1.0 and "truncated_fraction" in summary,
      f"the truncated fraction must be measured and recorded — got {TRUNCATED_FRACTION:.4f}. A "
      f"model evaluated at max_length=128 without that number reported is a result with an "
      f"unstated caveat",
      f"يجب أن تُقاس نسبة المقطوع وتُسجَّل — والناتج {TRUNCATED_FRACTION:.4f}. فالنموذج المُقيَّم "
      f"عند `max_length=128` بلا عرض ذلك الرقم نتيجةٌ بتحفّظٍ غير مُعلَن")

check(len(LOSS_LOG) >= 3 and LOSS_LOG[-1][1] < LOSS_LOG[0][1],
      f"the mean training loss must be logged and the last window must sit below the first — "
      f"{len(LOSS_LOG)} windows, from {LOSS_LOG[0][1]:.4f} to {LOSS_LOG[-1][1]:.4f}. This is a "
      f"trend check on window means, not on single batches: individual steps here swing by half a "
      f"nat and would fail this check on a run that trained perfectly well",
      f"يجب أن تُسجَّل خسارة التدريب المتوسطة وأن تجلس النافذة الأخيرة دون الأولى — "
      f"{len(LOSS_LOG)} نافذة، من {LOSS_LOG[0][1]:.4f} إلى {LOSS_LOG[-1][1]:.4f}. وهذا فحص "
      f"اتّجاهٍ على متوسطات النوافذ لا على دفعاتٍ منفردة: فالخطوات المنفردة هنا تتأرجح نصف نات "
      f"وكانت تُفشِل هذا الفحص على تشغيلةٍ تدرّبت تدرّبًا سليمًا")

comparison_text = COMPARISON_PATH.read_text(encoding="utf-8")
columns = ["accuracy", "F1", "fit time", "inference / 1,000", "on disk"]
missing = [column for column in columns if column not in comparison_text]
check(not missing,
      f"comparison.md must carry all five columns for both models — missing {missing}. Both time "
      f"columns especially: a comparison without cost is the failure this lab exists to prevent",
      f"يجب أن يحمل `comparison.md` الأعمدة الخمسة للنموذجين — والناقص {missing}. وعمودا الزمن "
      f"خصوصًا: فالمقارنة بلا كلفة هي الفشل الذي وُجد هذا المعمل لمنعه")

check(all(k in NEGATION_RESULTS for k in ("bert", "tfidf", "bert_correct", "tfidf_correct"))
      and str(NEGATION_RESULTS["bert_correct"]) in comparison_text,
      f"the negation pair result must be recorded for both models and appear in comparison.md — "
      f"BERT {NEGATION_RESULTS['bert']}, TF-IDF {NEGATION_RESULTS['tfidf']}",
      f"يجب أن تُسجَّل نتيجة زوج النفي للنموذجين وأن تظهر في `comparison.md` — BERT "
      f"{NEGATION_RESULTS['bert']}، وTF-IDF {NEGATION_RESULTS['tfidf']}")

check(0.5 < BERT_ACCURACY < 1.0 and f"{BERT_ACCURACY:.4f}" in comparison_text,
      f"BERT's accuracy must be above chance and must be written to the file even if it lost — "
      f"got {BERT_ACCURACY:.4f} against TF-IDF's {TFIDF_ACCURACY:.4f}. A notebook that silently "
      f"drops an unflattering result is worse than one that never ran",
      f"يجب أن تكون دقّة BERT فوق الحظّ وأن تُكتب في الملف وإن خسر — والناتج {BERT_ACCURACY:.4f} "
      f"مقابل {TFIDF_ACCURACY:.4f} لـTF-IDF. فالدفتر الذي يُسقط نتيجةً غير مُرضية صامتًا أسوأ من "
      f"دفترٍ لم يعمل قطّ")

report()

## What's next

**Week 6 — Representation learning and self-supervision.** You spent this week making text into
vectors. Next week does the same thing for images, and without labels: an autoencoder, a masked
autoencoder that hides 75% of the picture, DINO, a linear probe, and CLIP — which puts images and
text in one shared space and makes this week's cosine work across both.

The block you built on Thursday is the same block. `ViTMAE` and `DINOv2` are transformer encoders
over image patches instead of subword tokens, and the shape discipline transfers exactly.

**A5 is issued today**, and it is this lab written up: `comparison.md`, D2's `similarity_report.md`,
and the recommendation paragraph defended against the constraint. The numbers are already on your
disk.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع السادس — تعلّم التمثيلات والإشراف الذاتي.** أنفقت هذا الأسبوع في تحويل النص متّجهات.
ويفعل الأسبوع القادم الشيء نفسه بالصور وبلا تسميات: مُرمِّزٌ تلقائي، ومُرمِّزٌ تلقائي مُقنَّع يُخفي
٧٥٪ من الصورة، وDINO، ومِسبارٌ خطّي، وCLIP — الذي يضع الصور والنص في فضاءٍ مشترك واحد ويجعل جيب
تمام هذا الأسبوع يعمل عليهما معًا.

والكتلة التي بنيتها الخميس هي الكتلة نفسها. فـ`ViTMAE` و`DINOv2` مُرمِّزات محوّلات على رِقاع صورٍ بدل
رموزٍ جزئية، وانضباط الأشكال ينتقل بحاله.

**ويُطرح التكليف الخامس اليوم**، وهو هذا المعمل مكتوبًا: `comparison.md`، و`similarity_report.md`
من اليوم الثاني، وفقرة التوصية مُدافَعًا عنها مقابل القيد. والأرقام على قرصك أصلًا.

</div>
